## PMXT API -> Kafka 原始数据链路测试

本部分用于验证当前目标的数据流前半段：

- 从 PMXT API 获取 `fetch_markets` 原始数据
- 以 JSON 消息写入 Kafka
- 从 Kafka 消费并检查消息结构


In [1]:
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

# 让 notebook 里可以 import repo 内的 app 包
repo_root = Path.cwd()
repo_root = repo_root.parent
if (repo_root / "app").exists():
    sys.path.insert(0, str(repo_root))

from app.config import settings as app_settings
from app.clients.kafka_client import KafkaConsumerClient

print("Kafka bootstrap servers:", app_settings.KAFKA_BOOTSTRAP_SERVERS)
print("Kafka raw markets topic:", app_settings.KAFKA_TOPIC_POLYMARKET_MARKETS_RAW)


Kafka bootstrap servers: localhost:9092
Kafka raw markets topic: polymarket.markets.raw


In [20]:
from datetime import datetime

DO_INGEST = False  # 是否要手动进行一次写入Kafka的操作
DO_CONSUME_CHECK = True  # 读取 Kafka 检查消息内容

if DO_INGEST:
    from app.services.ingestion_service import ingestion_service

    # 默认参数：sort='volume', limit=2000，但是这里测试只做5条
    published = ingestion_service.publish_polymarket_markets_to_kafka(
        # query="Trump",
        limit=5,
        sort="volume",
    )
    print("Published market messages to Kafka:", published)
else:
    print("Skip ingestion (DO_INGEST=False).")


if DO_CONSUME_CHECK:
    topic = app_settings.KAFKA_TOPIC_POLYMARKET_MARKETS_RAW
    group_id = f"notebook.kafka.check.latest.{datetime.utcnow().strftime('%H%M%S')}"

    with KafkaConsumerClient(
        topic=topic,
        group_id=group_id,
        auto_offset_reset="latest",
        enable_auto_commit=False,
        consumer_timeout_ms=3000,
    ) as consumer:
        result = consumer.consume(max_messages_per_partition=500, max_workers=3)

    rows = []
    for partition, messages in result.items():
        for msg in messages:
            rows.append(
                {
                    "partition": partition,
                    "market_id": msg.get("market_id"),
                    "source": msg.get("source"),
                    "exchange": msg.get("exchange"),
                    "captured_at": msg.get("captured_at"),
                    "has_payload": "payload" in msg,
                    "payload": msg.get("payload")
                }
            )

    print(f"Consumer group: {group_id}")
    print(f"Consumed messages: {len(rows)}")
    if rows:
        display(pd.DataFrame(rows).head(20))
    else:
        print("No messages consumed. Check broker/topic connectivity or DO_INGEST=True.")

Skip ingestion (DO_INGEST=False).
2026-03-21 16:22:25 | INFO | app.clients.kafka_client | Initializing KafkaConsumerClient (topic=polymarket.markets.raw, group_id=notebook.kafka.check.latest.082225)
2026-03-21 16:22:25 | INFO | app.clients.kafka_client | Opening Kafka consumer connection (topic=polymarket.markets.raw, group_id=notebook.kafka.check.latest.082225)


C:\Users\Lily\AppData\Local\Temp\ipykernel_18988\2476294516.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  group_id = f"notebook.kafka.check.latest.{datetime.utcnow().strftime('%H%M%S')}"


2026-03-21 16:22:25 | INFO | app.clients.kafka_client | Starting scheduled bounded consumption (topic=polymarket.markets.raw, partitions=[0, 1, 2], max_per_partition=500)
2026-03-21 16:22:25 | INFO | app.clients.kafka_client | Partition 0 bounded consume (start=0, stop=500, end=12180)
2026-03-21 16:22:25 | INFO | app.clients.kafka_client | Partition 2 bounded consume (start=0, stop=500, end=13089)
2026-03-21 16:22:25 | INFO | app.clients.kafka_client | Partition 1 bounded consume (start=0, stop=500, end=12755)
2026-03-21 16:22:26 | INFO | app.clients.kafka_client | Partition 2 consume completed (500 messages)
2026-03-21 16:22:26 | INFO | app.clients.kafka_client | Partition 0 consume completed (500 messages)
2026-03-21 16:22:26 | INFO | app.clients.kafka_client | Partition 1 consume completed (500 messages)
2026-03-21 16:22:26 | INFO | app.clients.kafka_client | Scheduled bounded consumption completed (topic=polymarket.markets.raw, total_partitions=3, total_messages=1500)
2026-03-21 16

,partition,market_id,source,exchange,captured_at,has_payload,payload
0,2,561973,pmxt,polymarket,2026-03-21T07:35:51.406617Z,True,"{'market_id': '561973', 'title': 'Republican P..."
1,2,561973,pmxt,polymarket,2026-03-21T07:36:44.250203Z,True,"{'market_id': '561973', 'title': 'Republican P..."
2,2,561973,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '561973', 'title': 'Republican P..."
3,2,562006,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '562006', 'title': 'Republican P..."
4,2,561998,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '561998', 'title': 'Republican P..."
5,2,561973,pmxt,polymarket,2026-03-21T07:45:04.589695Z,True,"{'market_id': '561973', 'title': 'Republican P..."
6,2,562006,pmxt,polymarket,2026-03-21T07:45:04.589695Z,True,"{'market_id': '562006', 'title': 'Republican P..."
7,2,561998,pmxt,polymarket,2026-03-21T07:45:04.589695Z,True,"{'market_id': '561998', 'title': 'Republican P..."
8,2,559664,pmxt,polymarket,2026-03-21T07:47:14.008469Z,True,"{'market_id': '559664', 'title': 'Democratic P..."
9,2,561996,pmxt,polymarket,2026-03-21T07:47:14.008469Z,True,"{'market_id': '561996', 'title': 'Republican P..."


In [16]:
# 单条数据示例
pd.DataFrame(rows)['payload'].iloc[0]

{'market_id': '562007',
 'title': 'Republican Presidential Nominee 2028 - Will Joe Kent win the 2028 Republican presidential nomination?',
 'outcomes': [{'outcome_id': '79109138136144804495019974949003266649145803384760006308278371005319168843129',
   'label': 'Joe Kent',
   'price': 0.0075,
   'price_change_24h': 0,
   'metadata': {'clobTokenId': '79109138136144804495019974949003266649145803384760006308278371005319168843129'},
   'market_id': '562007'},
  {'outcome_id': '65030375464178446496570525340832652143777619585997981475842311804446352421562',
   'label': 'Not Joe Kent',
   'price': 0.9925,
   'price_change_24h': 0,
   'metadata': {'clobTokenId': '65030375464178446496570525340832652143777619585997981475842311804446352421562'},
   'market_id': '562007'}],
 'volume_24h': 608979.413361,
 'liquidity': 575368.27402,
 'url': 'https://polymarket.com/event/republican-presidential-nominee-2028',
 'description': 'This market will resolve to “Yes” if the named individual wins and accepts t